### Variables

In [960]:
import os
import pandas as pd
import psycopg2
from pgvector.psycopg2 import register_vector
import numpy as np
from sklearn.cluster import DBSCAN, KMeans
from sklearn.metrics.pairwise import cosine_distances

RESUME_ID = "72b39379-e2da-4b28-9102-a32b77eacd97"
SOFT_SKILLS_SIMILARITY_THRESHOLD = 0.66
HARD_SKILLS_SIMILARITY_THRESHOLD = 0.66
SOFT_SKILLS_WEIGHT_COLUMN_INDEX = 3
SOFT_SKILLS_STRING_COLUMN_INDEX = 4
HARD_SKILLS_STRING_COLUMN_INDEX = 5
HARD_SKILLS_WEIGHT_COLUMN_INDEX = 3

LLM_MODEL_VECTOR_DIMENSIONS = 3072
DB_NAME = "market_fit"
DB_HOST = "localhost"
db_user = os.getenv("DB_USER")
db_pw = os.getenv("DB_PASSWORD")
api_key = os.getenv('GEMINI_API_KEY')
JOB_POSTINGS_TABLE = "job_postings_general"
HARD_SKILLS_TABLE = "hard_skills"
SOFT_SKILLS_TABLE = "soft_skills"
RESUMES_TABLE = "resumes"
INDUSTRIES_TABLE = "work_industries"
CANDIDATE_HARD_SKILLS_TABLE = "candidate_hard_skills"
CANDIDATE_SOFT_SKILLS_TABLE = "candidate_soft_skills"

### Functions Definitions

In [961]:
def db_connect() -> psycopg2.extensions.connection:
    conn = psycopg2.connect(
        host=DB_HOST, dbname=DB_NAME, user=db_user, password=db_pw
    )
    register_vector(conn)
    return conn

def get_resume(resume_id: str) -> pd.DataFrame:
    with db_connect() as conn:
        with conn.cursor() as cur:
            cur.execute(f"SELECT * FROM {RESUMES_TABLE} WHERE id = %s", (resume_id,))
            rows = cur.fetchall()
            cols = [desc[0] for desc in cur.description]
    return pd.DataFrame(rows, columns=cols)

def filter_job_postings(industries: list) -> pd.DataFrame:
    filter_string = f"WHERE ai_industries::TEXT[] && ARRAY[{industries}]"
    with db_connect() as conn:
        with conn.cursor() as cur:
            cur.execute(f"SELECT id, title, description FROM {JOB_POSTINGS_TABLE} {filter_string}")
            rows = cur.fetchall()
            cols = [desc[0] for desc in cur.description]
    return pd.DataFrame(rows, columns=cols)

def get_position_skills(jobs_ids: list, table_name: str) -> pd.DataFrame:
    with db_connect() as conn:
        with conn.cursor() as cur:
            cur.execute(f"SELECT * FROM {table_name} WHERE job_id = ANY(%s)", (jobs_ids,))
            rows = cur.fetchall()
            cols = [desc[0] for desc in cur.description]
    return pd.DataFrame(rows, columns=cols)

def get_candidate_skills(resume_id: str, table_name: str) -> pd.DataFrame:
    with db_connect() as conn:
        with conn.cursor() as cur:
            cur.execute(f"SELECT * FROM {table_name} WHERE resume_id = '{resume_id}'")
            rows = cur.fetchall()
            cols = [desc[0] for desc in cur.description]
    return pd.DataFrame(rows, columns=cols)    

def cosine_similarities_matrix(query: np.ndarray, matrix: np.ndarray) -> np.ndarray:
    zero_mask = np.all(matrix == 0, axis=-1)  
    query_norm = np.linalg.norm(query)
    row_norms = np.linalg.norm(matrix, axis=-1)  
    dot_products = matrix @ query
    similarities = dot_products / (row_norms * query_norm + 1e-10)
    similarities[zero_mask] = 0.0
    return similarities  

def get_compliance_mask(
        compliance_type: str,
        cosine_similarities: np.ndarray, 
        threshold: float,
        weight_matrix: np.ndarray = None, 
        weight: float = 0):    
    if compliance_type == "COMPLIANT":
        binary_mask = (cosine_similarities > threshold).astype(np.int8)
    elif compliance_type == "NONCOMPLIANT":
        binary_mask = (cosine_similarities <= threshold).astype(np.int8)    
    elif compliance_type == "IDEAL" and weight != 0 and weight_matrix is not None:
        minimum_weights_matrix = np.where(binary_mask == 1, weight_matrix, 0)    
        binary_mask = ((weight >= minimum_weights_matrix) & (binary_mask == 1)).astype(np.int8)
    return binary_mask

def create_mapping_matrix(count_list: list):
    nrows = len(count_list)
    ncols = max(count_list)
    matrix = np.zeros((nrows, ncols), dtype=np.int8)
    for i, count in enumerate(count_list):
        matrix[i, :count] = 1
    return matrix     

def analyze_market(market_obj: MarketSkillsMatrix, candidate_skills_df: pd.DataFrame, skills_type: str) -> None:
    weight_column_index = SOFT_SKILLS_WEIGHT_COLUMN_INDEX if skills_type == "soft" else HARD_SKILLS_WEIGHT_COLUMN_INDEX
    string_column_index = SOFT_SKILLS_STRING_COLUMN_INDEX if skills_type == "soft" else HARD_SKILLS_STRING_COLUMN_INDEX
    threshold = SOFT_SKILLS_SIMILARITY_THRESHOLD if skills_type == "soft" else HARD_SKILLS_SIMILARITY_THRESHOLD
    skills_count = candidate_skills_df.shape[0]

    for i in range(0, skills_count):
        weight = candidate_skills_df.iloc[(i, weight_column_index)] 
        skill_embedding = candidate_skills_df.iloc[(i, string_column_index) ] 
        cosine_similarities = cosine_similarities_matrix(skill_embedding, market_obj.embedding_matrix)
        binary_mask = get_compliance_mask("COMPLIANT", cosine_similarities, threshold)        
        market_obj.accumulate_matches(binary_mask)
        market_obj.accumulate_weighted_matches(weight, binary_mask)

def build_padded_matrix(
        df: pd.DataFrame,
        column_name: str,
        length: int,
        pad_value,
        dtype=None,
    ) -> np.ndarray:
        rows = [
            np.pad(
                array=group[column_name].values,
                pad_width=(0, length - len(group)),
                constant_values=pad_value,
            )
            for _, group in df.groupby("index")
        ]
        return np.array(rows, dtype=dtype)

def build_embedding_matrix(
    df: pd.DataFrame, max_skills: int
    ) -> np.ndarray:
        zero_vector = np.zeros(LLM_MODEL_VECTOR_DIMENSIONS)
        rows = [
            np.vstack(
                list(group["embedding"].values)
                + [zero_vector] * (max_skills - len(group))
            )
            for _, group in df.groupby("index")
        ]
        return np.array(rows, dtype=np.float32)


class MarketSkillsMatrix:
    _MATCH_SCORE_THRESHOLD = 1

    def __init__(self, skill_type: str, jobs_df: pd.DataFrame):
        if skill_type not in ("hard", "soft"):
            raise ValueError(f"skill_type must be 'hard' or 'soft', got '{skill_type}'")
        if jobs_df.empty:
            raise ValueError("jobs_df cannot be empty")

        self.skill_type = skill_type

        # Populated by _initialize_matrices
        self.string_matrix: np.ndarray = None      # (n_jobs, max_skills) skill descriptions
        self.weight_matrix: np.ndarray = None      # (n_jobs, max_skills) skill weights
        self.embedding_matrix: np.ndarray = None   # (n_jobs, max_skills, vector_dim) embeddings
        self.count_by_index: list[int] = []        # count of skills per job
        self.job_id_by_index: list = []            # job_id mapped to matrix row index

        # Populated after combine() / weight_against() calls
        self._match_score_matrix: np.ndarray = None     # raw match counts per (job, skill) cell
        self._weighted_match_matrix: np.ndarray = None  # weight-qualified match counts

        self._initialize_matrices(jobs_df)

    def _initialize_matrices(self, jobs_df: pd.DataFrame) -> None:
        matching_jobs_ids = jobs_df["id"].tolist()
        table = SOFT_SKILLS_TABLE if self.skill_type == "soft" else HARD_SKILLS_TABLE
        skills_df = get_position_skills(matching_jobs_ids, table).sort_values("job_id")

        skills_df["index"] = skills_df.groupby("job_id").ngroup()
        max_skills_per_job = skills_df.groupby("index").size().max()

        self.string_matrix = build_padded_matrix(
            skills_df, "skill_description", max_skills_per_job, pad_value=""
        )
        self.weight_matrix = build_padded_matrix(
            skills_df, "weight", max_skills_per_job, pad_value=0, dtype=np.float32
        )
        self.embedding_matrix = build_embedding_matrix(
            skills_df, max_skills_per_job
        )
        self.count_by_index = (
            skills_df["index"].value_counts().sort_index().tolist()
        )
        self.job_id_by_index = (
            skills_df[["index", "job_id"]].drop_duplicates()["job_id"].tolist()
        )
        self._match_score_matrix = create_mapping_matrix(self.count_by_index)
        self._weighted_match_matrix = np.zeros_like(self._match_score_matrix)

    def accumulate_matches(self, match_array: np.ndarray) -> None:
        """Add a match score array into the running match score matrix."""
        if match_array.shape != self._match_score_matrix.shape:
            raise ValueError(
                f"match_array shape {match_array.shape} does not match "
                f"expected {self._match_score_matrix.shape}"
            )
        self._match_score_matrix += match_array

    def accumulate_weighted_matches(
        self, skill_weight: float, binary_mask: np.ndarray
    ) -> None:
        """Record which job skills are met by a candidate skill at the given weight."""
        if binary_mask.shape != self.weight_matrix.shape:
            raise ValueError(
                f"binary_mask shape {binary_mask.shape} does not match "
                f"weight_matrix shape {self.weight_matrix.shape}"
            )
        candidate_weight_mask = binary_mask * skill_weight
        weight_qualified = (candidate_weight_mask >= self.weight_matrix) & (self.weight_matrix != 0)
        self._weighted_match_matrix += weight_qualified.astype(np.int8)

    def get_minimum_compliance_by_job(self) -> list[float]:
        """
        Percentage of a job's skills matched above the minimum score threshold,
        per job index. A cell qualifies if its match score exceeds _MATCH_SCORE_THRESHOLD.
        """
        qualifying = (self._match_score_matrix > self._MATCH_SCORE_THRESHOLD).sum(axis=1)
        return [round(100 * a / b, 2) for a, b in zip(qualifying.tolist(), self.count_by_index)]

    def get_ideal_compliance_by_job(self) -> list[float]:
        """
        Percentage of a job's skills met at or above their required weight,
        per job index.
        """
        qualifying = (self._weighted_match_matrix != 0).sum(axis=1)
        return [round(100 * a / b, 2) for a, b in zip(qualifying.tolist(), self.count_by_index)]

### Get Candidate Info

In [962]:
RESUME_ID = "38e1e96d-f850-46f0-81f6-97d8ac11cf8a"  # REMOVE  
resume_df = get_resume(RESUME_ID)
candidate_industries = resume_df["industries"][0]

candidate_hard_skills_df = get_candidate_skills(RESUME_ID, CANDIDATE_HARD_SKILLS_TABLE)
candidate_soft_skills_df = get_candidate_skills(RESUME_ID, CANDIDATE_SOFT_SKILLS_TABLE)

jobs_df = filter_job_postings(candidate_industries)

soft_market = MarketSkillsMatrix("soft", jobs_df.copy())
hard_market = MarketSkillsMatrix("hard", jobs_df.copy())

Find candidate's best matches for each skill  
Consider soft and hard skills

In [963]:
def get_market_analysis_results(market_obj: MarketSkillsMatrix) -> list[dict]:
    compliance_by_job = market_obj.get_minimum_compliance_by_job()
    ideal_compliance_by_job = market_obj.get_ideal_compliance_by_job()
    noncompliance_mask = market_obj._match_score_matrix == 1

    return [
        {
            "job_id": job_id,
            "job_index": i,
            "minimum_compliance_pct": compliance_pct,
            "ideal_compliance_pct": ideal_compliance_pct,
            "nonmatched_skills_count": int(noncompliance_mask[i].sum()),
            "nonmatched_skills": list(set(market_obj.string_matrix[i][noncompliance_mask[i]])),
            "similarity_matched_skills": get_similarity_matched_skills(market_obj, i),
            "similarity_match_scores": get_similarity_matched_skills(market_obj, i, with_scores=True),
            "not_ideal_skills": list(set(
                    market_obj.string_matrix[i][
                        (market_obj._weighted_match_matrix[i] == 0) &
                        (market_obj.string_matrix[i] != "")
                    ]
                ))
        }
        for i, (job_id, compliance_pct, ideal_compliance_pct) in enumerate(
            zip(market_obj.job_id_by_index, compliance_by_job, ideal_compliance_by_job)
        )
    ]

def get_similarity_matched_skills(market_obj: MarketSkillsMatrix, job_index: int, top_n: int = 5, with_scores: bool = False):
    scores = market_obj._match_score_matrix[job_index]
    descriptions = market_obj.string_matrix[job_index]
    ranked = sorted(
        ((desc, int(score)) for desc, score in zip(descriptions, scores) if desc != "" and score > 0),
        key=lambda x: x[1],
        reverse=True,
    )
    ranked = ranked[:top_n]
    return ranked if with_scores else [desc for desc, _ in ranked]

def print_analysis(market_obj: MarketSkillsMatrix, analysis: list[dict]) -> None:
    sorted_analysis = sorted(analysis, key=lambda e: e["minimum_compliance_pct"], reverse=True)
    sorted_counts = [market_obj.count_by_index[market_obj.job_id_by_index.index(e["job_id"])] for e in sorted_analysis]

    print("Count of required skills by job:", sorted_counts)
    print()
    for entry in sorted_analysis:
        print(
            f"[{entry['job_id']} | {entry['job_index']}]\n"
            f"  minimum: {entry['minimum_compliance_pct']}% | ideal: {entry['ideal_compliance_pct']}%\n"
            f"  present skills matches ({len(entry['similarity_matched_skills'])}):  {entry['similarity_matched_skills']}\n"
            f"  insufficient proficiency ({len(entry['not_ideal_skills'])}): {entry['not_ideal_skills']}\n"
            f"  nonmatched ({entry['nonmatched_skills_count']}): {entry['nonmatched_skills']}"
        )

def print_popular_similarity_matches(analysis: list[dict]) -> None:
    skill_counts: dict[str, int] = {}
    for entry in analysis:
        for skill, score in entry["similarity_match_scores"]:
            skill_counts[skill] = skill_counts.get(skill, 0) + score

    print("Most popular similarity matches (across all jobs):")
    for skill, count in sorted(skill_counts.items(), key=lambda x: x[1], reverse=True):
        print(f"  {skill}: {count}")

In [964]:
analyze_market(soft_market, candidate_soft_skills_df, "soft")
market_soft_skills_analysis = get_market_analysis_results(soft_market)
analyze_market(hard_market, candidate_hard_skills_df, "hard")
market_hard_skills_analysis = get_market_analysis_results(hard_market)

In [965]:
print_analysis(soft_market, market_soft_skills_analysis)
print('=========================================================')
print_analysis(hard_market, market_hard_skills_analysis)

Count of required skills by job: [8, 7, 7, 5, 5, 8, 8, 6, 9, 13, 12, 16, 8, 20, 25, 11, 11, 13, 6, 7, 12, 17, 22, 7]

[2116479190 | 0]
  minimum: 87.5% | ideal: 25.0%
  present skills matches (5):  ['Communication Skills', 'Organizational Skills', 'Remote Work Adaptability', 'Independent Work Ethic', 'Verbal Communication']
  insufficient proficiency (6): ['Independent Work Ethic', 'Process Improvement Identification', 'Verbal Communication', 'Written Communication', 'Organizational Skills', 'Communication Skills']
  nonmatched (1): ['Process Improvement Identification']
[2122684182 | 15]
  minimum: 85.71% | ideal: 57.14%
  present skills matches (5):  ['Communication (Verbal)', 'Accuracy', 'Detail-Oriented', 'Communication (Written)', 'Independent Work Ethic']
  insufficient proficiency (3): ['Collaboration', 'Communication (Verbal)', 'Communication (Written)']
  nonmatched (1): ['Collaboration']
[2123419182 | 17]
  minimum: 85.71% | ideal: 42.86%
  present skills matches (5):  ['Accu

In [966]:
print_popular_similarity_matches(market_soft_skills_analysis)
print_popular_similarity_matches(market_hard_skills_analysis)

Most popular similarity matches (across all jobs):
  Analytical Skills: 15
  Communication Skills: 12
  Organizational Skills: 12
  Attention to Detail: 12
  Accuracy: 12
  Adaptability: 12
  Detail-Oriented: 9
  Problem Solving: 8
  Problem-Solving Skills: 8
  Independent Work Ethic: 6
  Organizational skills: 5
  Problem Solving Skills: 5
  Verbal Communication: 4
  Communication: 4
  Time Management: 4
  Attention to detail: 4
  Systems thinking: 4
  Clear communication: 4
  Analytical Thinking: 4
  Analytical mindset: 4
  Problem-solving skills: 4
  Financial Accuracy: 3
  Analytical Ability: 3
  Detail-Orientation: 3
  Process-Driven: 3
  Problem-solving: 3
  Accuracy focus: 3
  Efficiency focus: 3
  Documentation skills: 3
  Analytical skills: 3
  Time-management abilities: 3
  Accurate reporting: 3
  Detail Orientation: 3
  Interdepartmental Communication: 3
  Communication (Verbal): 3
  Presentation Skills: 3
  Precision: 3
  Interpersonal Skills: 3
  Collaboration Skills: 3
  

### Test clusterization

In [967]:
noncompliance_mask = hard_market._match_score_matrix == 1

missing_skills_matrix = np.where(
    noncompliance_mask[:, :, np.newaxis],
    hard_market.embedding_matrix,
    np.nan
)

# Flatten and remove nans
flat = missing_skills_matrix.reshape(-1, LLM_MODEL_VECTOR_DIMENSIONS)
valid_mask = ~np.isnan(flat).any(axis=1)
missing_skills_matrix = flat[valid_mask]

# Find good eps
distances = cosine_distances(missing_skills_matrix)
nearest = np.sort(distances, axis=1)[:, 1]
eps = np.percentile(nearest, 50)
print(f"Using eps: {eps}")

# DBSCAN to find k
labels_dbscan = DBSCAN(eps=eps, min_samples=2, metric='cosine').fit_predict(missing_skills_matrix)
k = len(set(labels_dbscan)) - (1 if -1 in labels_dbscan else 0)
print(f"DBSCAN found {k} clusters")

# KMeans with k
kmeans = KMeans(n_clusters=k)
labels_kmeans = kmeans.fit_predict(missing_skills_matrix)

# Group vectors
grouped = {}
outlier_count = 0
for embedding, dbscan_label, kmeans_label in zip(missing_skills_matrix, labels_dbscan, labels_kmeans):
    if dbscan_label == -1:
        grouped[f"outlier_{outlier_count}"] = [embedding]
        outlier_count += 1
    else:
        grouped.setdefault(kmeans_label, []).append(embedding)

grouped = list(grouped.values())
print(f"Total groups: {len(grouped)}")

Using eps: 0.23277661204338074
DBSCAN found 52 clusters
Total groups: 203


In [ ]:
# Extract descriptions in the same order as the flattened embeddings
flat_descriptions = hard_market.string_matrix[noncompliance_mask]

# Group descriptions by kmeans label, handling outliers
clusters: dict = {}
outlier_count = 0
for desc, dbscan_label, kmeans_label in zip(flat_descriptions, labels_dbscan, labels_kmeans):
    if dbscan_label == -1:
        clusters[f"outlier_{outlier_count}"] = [desc]
        outlier_count += 1
    else:
        clusters.setdefault(kmeans_label, []).append(desc)
        
def print_popular_nonmatched_clusters(clusters: dict) -> None:
    print("Nonmatched skill clusters:")
    for label, skills in sorted(clusters.items(), key=lambda x: len(x[1]), reverse=True):
        unique = list(set(skills))
        print(f"  cluster {label} ({len(skills)} skills): {unique}")
        
print_popular_nonmatched_clusters(clusters)

Nonmatched skill clusters:
  cluster 1 (11 skills): ['Month-end close', 'Month-end close processes', 'Month-End Close', 'Month-End Closing']
  cluster 8 (11 skills): ['Fluent English', 'Fluent English (B2+)', 'Fluent Spanish']
  cluster 11 (8 skills): ['Financial management tools', 'Financial controls', 'Expense Tracking', 'Financial Information Reliability', 'Expense management tools', 'Controlling Routines Experience', 'Cost Control', 'Financial Routines Experience']
  cluster 20 (7 skills): ['ABVTEX Compliance', 'Legal Compliance', 'Compliance', 'Regulatory Compliance', 'Tax Compliance']
  cluster 6 (6 skills): ["Bachelor's degree in Finance", "Bachelor's degree in Accounting or related field", 'Accounting Degree', "Bachelor's degree in Accounting"]
  cluster 13 (5 skills): ['Budgeting']
  cluster 26 (5 skills): ['U.S. GAAP', 'US Accounting Standards', 'US business structures', 'U.S. business experience']
  cluster 7 (5 skills): ['English', 'Portuguese']
  cluster 4 (5 skills): ['Ac